In [1]:
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz
import glob
import unidecode
import re

path_in = '../../../data/00-map/capacity/'
path_out = '../../../data/00-map/capacity/final/'

def normalize_name2(name):
    if pd.isnull(name):
        return ""
    name = name.lower().strip()                      # Lowercase and trim
    name = unidecode.unidecode(name)                 # Remove accents
    name = name.replace('*', '')
    name = name.replace(':', '')
    name = re.sub(r':?selected:?', '', name) 
    name = name.replace('"', '')
    name = name.replace('-', '')
    name = name.replace('{', '').replace('}', '')
    name = name.replace('.', ' ')
    name = name.replace('(', '').replace(')', '')
    name = ' '.join(name.split())
    name = name.strip()
    return name


import re
# Functions for fuzzy
def clean_numeric_column(series):
    """
    Cleans a pandas Series with numeric values that may contain 
    commas, periods, or non-numeric characters.
    
    Returns a Series with float values or NaN.
    """
    def clean_value(val):
        if pd.isnull(val):
            return 0
        val = str(val)
        val = re.sub(r'[^\d.,]', '', val)        # Keep only digits, commas, periods
        val = val.replace(',', '')               # Remove commas
        val = val.replace(':', '')
        val = val.replace('$', '')  
        val = val.replace(' ', '')               # Remove spaces
        try:
            return float(val)
        except:
            return None

    return series.apply(clean_value)


mexican_states = [
    "aguascalientes", "baja california", "baja california sur", "campeche", "distrito federal",
    "coahuila", "colima", "chiapas", "chihuahua", "durango", "guanajuato", "guerrero",
    "hidalgo", "jalisco", "mexico", "michoacan", "morelos", "nayarit", "nuevo leon",
    "oaxaca", "puebla", "queretaro", "quintana roo", "san luis potosi", "sinaloa",
    "sonora", "tabasco", "tamaulipas", "tlaxcala", "veracruz", "yucatan", "zacatecas", 
    "durango(r)", "coahuila de zaragoza", "michoacan de ocampo", "queretaro de arteaga", 
    "total", "complejo penitenciario islas marias"
]

states_list = [normalize_name2(s) for s in mexican_states]

federal = ['cefereso', 'federal', 'ceferepsi']

# Join them into a regex pattern
pattern = '|'.join(federal)

In [2]:
dictio = pd.read_excel(f'{path_in}geolocate/deduplicated_geolocated_manual.xlsx')

dictio['prison_id'] = dictio['CVE_ENT'].astype(str)+'_'+dictio['cluster_manual'].str.replace(' ', '_')

# Build dictionary

# Initialize name to id mapping
name_to_id = {}

# Normalize names and fill the dictionary
for _, row in dictio.iterrows():
    id_val = row['prison_id']
    for col in ['center_name1', 'center_name2', 'center_name3', 
                'center_name4', 'center_name5', 'center_name6', 
                'center_name7']:  # Add more if you have more name columns
        name = normalize_name2(row[col])
        if name:  # Skip empty strings
            name_to_id[name] = id_val

# List of all normalized names for matching
all_names = list(name_to_id.keys())

def safe_fuzzy_match(x):
    result = process.extractOne(x, all_names, scorer=fuzz.token_sort_ratio)
    if result is not None:
        return pd.Series([result[0], result[1]])  # name, score
    else:
        return pd.Series([None, None])

In [3]:
for year in range(2000, 2015):
    panel = pd.DataFrame()
    panel2 = pd.DataFrame()
    for month in range(2, 13, 2):
        df = pd.read_excel(f'{path_in}raw/{year}/capacity_{month}_checked.xlsx')
        df['comun_clean'] = clean_numeric_column(df['comun'])
        df['federal_clean'] = clean_numeric_column(df['federal'])
        df['capacity_clean'] = clean_numeric_column(df['capacity'])
        df['total_clean'] = clean_numeric_column(df['total'])
        df['center_name_clean'] = df['center_name'].apply(normalize_name2)
        df[['matched_name', 'match_score']] = df['center_name_clean'].apply(safe_fuzzy_match)
        df['prison_id'] = df['matched_name'].map(name_to_id)

        df['is_federal'] = df['center_name_clean'].str.contains(pattern, case=False, na=False).astype(int)
        df['is_state'] = df['center_name_clean'].isin(states_list).astype(int)
        df_final = df[df['is_state'] == 0]
        df_final = df_final[df_final['is_federal'] == 0]
        df_final = df_final[['prison_id', 'center_name_clean', 'matched_name', 'match_score', 
                             'total', 'total_clean', 'capacity', 'capacity_clean', 
                             'comun', 'comun_clean', 'federal', 'federal_clean']].drop_duplicates('prison_id')
        test = df_final[df_final['match_score']<80]
        test['month'] = month
        test = test.drop_duplicates('center_name_clean').reset_index(drop=True)
        df_final = df_final[df_final['match_score']>80]
        merged = dictio[['prison_id', 'CVE_ENT', 
                         'lat_manual', 'long_manual']].drop_duplicates(['prison_id']).reset_index(drop=True)
        merged['year'] = year
        merged['month'] = month
        merged = merged.merge(df_final, on = ['prison_id'], how = 'left')
        panel = pd.concat([panel, merged]).reset_index(drop=True)
        panel2 = pd.concat([panel2, test]).reset_index(drop=True)
    panel.to_excel(f'{path_out}panel_capacity_{year}.xlsx', index = False)
    panel2.to_excel(f'{path_out}test_{year}.xlsx', index = False)

C:\Users\Dell\AppData\Local\Temp\ipykernel_700\3379686783.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test['month'] = month
C:\Users\Dell\AppData\Local\Temp\ipykernel_700\3379686783.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test['month'] = month
C:\Users\Dell\AppData\Local\Temp\ipykernel_700\3379686783.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documenta

In [4]:
# Finally: The full panel
final_panel = pd.DataFrame()

for i in range(2000, 2015):
    f = pd.read_excel(f'{path_out}panel_capacity_{i}.xlsx')
    final_panel = pd.concat([final_panel, f])

final_panel = final_panel.reset_index(drop=True)

final_panel = final_panel.drop(['capacity', 'total', 'federal', 'comun'], axis = 1)
final_panel.to_parquet(f'{path_out}final_panel_capacity.parquet.gzip', 
                       index = False, compression='gzip')

